# Stage 17D: selector resolution audit

always-selectorを凍結し、target-freeに固定抽出したsubsetでparticles・seeds・tracking stepsを増やして改善が維持されるか確認します。GPU・Kaggle提出は不要です。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'train').is_dir()
artifact_dir.mkdir(parents=True, exist_ok=True)
def run_checked(command):
    result = subprocess.run(command, cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.returncode != 0:
        print(result.stderr, flush=True)
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## 固定したStage 17B always-selector

Stage 17C gateは悪化したため使用しません。Stage 17Bのcut reportを直接監査します。

In [ ]:
stage17b_run = artifact_dir / 'stage17b_selector_replay_full_v001'
assert (stage17b_run / 'cut_report.parquet').is_file(), 'Run Notebook 380 Stage 17B first'
print('Stage 17B:', stage17b_run)

## Medium / high resolution subset audit

各fold × prefix fractionからSHA-256で固定抽出します。mediumは各stratum 8 cuts、highは2 cutsです。結果を見てsampleを変更しません。

In [ ]:
RUN_ID = 'stage17d_selector_resolution_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    run_checked([
        'uv', 'run', 'rogii-selector-resolution',
        '--config', 'configs/experiment/stage17d_selector_resolution.yaml',
        '--stage17b-run', str(stage17b_run), '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ])
else:
    print('Reusing completed run:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
{
    'stage17d_complete': summary['stage17d_complete'],
    'promoted_to_stage18': summary['promoted_to_stage18'],
    'profile_configs': summary['profile_configs'],
    'profile_reports': summary['profile_reports'],
    'gates': summary['gates'],
    'decision_if_promoted': summary['decision_if_promoted'],
    'next_step': summary['next_step'],
}

最後の辞書を共有してください。通過した場合、Stage 17はalways-selector controlとして完了し、Stage 18 branch retrievalへ移ります。